In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

S = "workspace.olist"
orders = spark.table(f"{S}.bronze_orders")
items = spark.table(f"{S}.bronze_order_items")
customers = spark.table(f"{S}.bronze_customers")
payments = spark.table(f"{S}.bronze_order_payments")
reviews = spark.table(f"{S}.bronze_order_reviews")
products = spark.table(f"{S}.bronze_products")
sellers = spark.table(f"{S}.bronze_sellers")
translation = spark.table(f"{S}.bronze_category_translation")

In [0]:
# Orders: dedupe, parse timestamps, derive delivery metrics
ts_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
           "order_delivered_customer_date", "order_estimated_delivery_date"]
orders_c = (orders.drop("ingested_at")
            .dropna(subset=["order_id", "customer_id"])
            .dropDuplicates(["order_id"]))
for c in ts_cols:
    orders_c = orders_c.withColumn(c, F.to_timestamp(c))
orders_c = (orders_c
    .withColumn("delivery_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    .withColumn("estimated_days", F.datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
    .withColumn("is_late",
        F.when(F.col("order_delivered_customer_date").isNull(), F.lit(None).cast("int"))
         .when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), F.lit(1))
         .otherwise(F.lit(0))))

# Items
items_c = (items.drop("ingested_at")
           .dropna(subset=["order_id", "order_item_id", "product_id", "seller_id"])
           .dropDuplicates(["order_id", "order_item_id"])
           .withColumn("price", F.col("price").cast("double"))
           .withColumn("freight_value", F.col("freight_value").cast("double")))

# Customers and sellers
customers_c = (customers.drop("ingested_at").dropna(subset=["customer_id"])
               .dropDuplicates(["customer_id"])
               .select("customer_id", "customer_unique_id", "customer_city", "customer_state"))
sellers_c = (sellers.drop("ingested_at").dropna(subset=["seller_id"])
             .dropDuplicates(["seller_id"])
             .select("seller_id", "seller_city", "seller_state"))

# Products with English category names
products_c = (products.drop("ingested_at").dropna(subset=["product_id"])
              .dropDuplicates(["product_id"])
              .join(translation.drop("ingested_at"), "product_category_name", "left")
              .withColumn("category_en", F.coalesce("product_category_name_english",
                                                    "product_category_name", F.lit("unknown")))
              .select("product_id", "category_en",
                      F.col("product_weight_g").cast("double").alias("product_weight_g"),
                      "product_photos_qty"))

# Payments aggregated per order (dominant payment type = largest payment)
w = Window.partitionBy("order_id").orderBy(F.col("payment_value").desc())
main_pay = (payments.drop("ingested_at")
            .withColumn("rn", F.row_number().over(w)).filter("rn = 1")
            .select("order_id", F.col("payment_type").alias("main_payment_type")))
pay_agg = (payments.drop("ingested_at").groupBy("order_id")
           .agg(F.sum("payment_value").alias("payment_total"),
                F.max("payment_installments").alias("max_installments"))
           .join(main_pay, "order_id", "left"))

# Reviews aggregated per order
review_agg = (reviews.drop("ingested_at").groupBy("order_id")
              .agg(F.avg(F.col("review_score").cast("double")).alias("avg_review_score")))

In [0]:
fact = (items_c
        .join(orders_c, "order_id", "inner")
        .join(customers_c, "customer_id", "left")
        .join(sellers_c, "seller_id", "left")
        .join(products_c, "product_id", "left")
        .join(pay_agg, "order_id", "left")
        .join(review_agg, "order_id", "left"))

tables = {"silver_orders": orders_c, "silver_order_items": items_c, "silver_customers": customers_c,
          "silver_sellers": sellers_c, "silver_products": products_c, "silver_order_fact": fact}
for name, df in tables.items():
    df.write.mode("overwrite").format("delta").saveAsTable(f"{S}.{name}")
    print(name, df.count())

silver_orders 99441
silver_order_items 112650
silver_customers 99441
silver_sellers 3095
silver_products 32951
silver_order_fact 112650


In [0]:
checks = []
def add(name, table, failing, total):
    checks.append((name, table, int(failing), int(total)))

add("duplicate_order_id_removed", "bronze_orders", orders.count() - orders.dropDuplicates(["order_id"]).count(), orders.count())
add("null_order_or_customer_id", "bronze_orders", orders.filter("order_id IS NULL OR customer_id IS NULL").count(), orders.count())
add("orphan_customer_id", "silver_orders", orders_c.join(customers_c, "customer_id", "left_anti").count(), orders_c.count())
add("orphan_order_id", "silver_order_items", items_c.join(orders_c, "order_id", "left_anti").count(), items_c.count())
add("orphan_product_id", "silver_order_items", items_c.join(products_c, "product_id", "left_anti").count(), items_c.count())
add("orphan_seller_id", "silver_order_items", items_c.join(sellers_c, "seller_id", "left_anti").count(), items_c.count())
add("nonpositive_price_or_negative_freight", "silver_order_items", items_c.filter("price <= 0 OR freight_value < 0").count(), items_c.count())
add("negative_delivery_days", "silver_orders", orders_c.filter("delivery_days < 0").count(), orders_c.count())
add("delivered_missing_delivery_date", "silver_orders", orders_c.filter("order_status = 'delivered' AND order_delivered_customer_date IS NULL").count(), orders_c.count())
add("unknown_product_category", "silver_products", products_c.filter("category_en = 'unknown'").count(), products_c.count())
add("null_customer_state", "silver_order_fact", fact.filter("customer_state IS NULL").count(), fact.count())
add("rows_lost_in_fact_join", "silver_order_fact", items_c.count() - fact.count(), items_c.count())

dq = (spark.createDataFrame(checks, "check_name string, table_name string, failing_rows long, total_rows long")
      .withColumn("status", F.when(F.col("failing_rows") == 0, "PASS").otherwise("FLAGGED"))
      .withColumn("run_ts", F.current_timestamp()))
dq.write.mode("overwrite").format("delta").saveAsTable(f"{S}.dq_results")
display(dq)

check_name,table_name,failing_rows,total_rows,status,run_ts
duplicate_order_id_removed,bronze_orders,0,99441,PASS,2026-10-28T09:40:00.122Z
null_order_or_customer_id,bronze_orders,0,99441,PASS,2026-10-28T09:40:00.122Z
orphan_customer_id,silver_orders,0,99441,PASS,2026-10-28T09:40:00.122Z
orphan_order_id,silver_order_items,0,112650,PASS,2026-10-28T09:40:00.122Z
orphan_product_id,silver_order_items,0,112650,PASS,2026-10-28T09:40:00.122Z
orphan_seller_id,silver_order_items,0,112650,PASS,2026-10-28T09:40:00.122Z
nonpositive_price_or_negative_freight,silver_order_items,0,112650,PASS,2026-10-28T09:40:00.122Z
negative_delivery_days,silver_orders,0,99441,PASS,2026-10-28T09:40:00.122Z
delivered_missing_delivery_date,silver_orders,8,99441,FLAGGED,2026-10-28T09:40:00.122Z
unknown_product_category,silver_products,610,32951,FLAGGED,2026-10-28T09:40:00.122Z


In [0]:
(spark.table(f"{S}.silver_order_fact").filter("order_status = 'delivered'")
 .select("order_id", "is_late").dropDuplicates(["order_id"])
 .agg(F.count("*").alias("delivered_orders"), F.round(F.avg("is_late") * 100, 2).alias("late_pct")).show())

+----------------+--------+
|delivered_orders|late_pct|
+----------------+--------+
|           96478|    8.11|
+----------------+--------+

